In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv


In [2]:
%%writefile my_preprocessing_classes.py
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder

class FraudDataCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, drop_threshold=0.9):
        self.drop_threshold = drop_threshold
        self.cols_to_remove = ['TransactionID', 'TransactionDT']
        self.high_nan_cols = []

    def fit(self, X, y=None):
        nan_shares = X.isnull().mean()
        self.high_nan_cols = nan_shares[nan_shares > self.drop_threshold].index.tolist()
        return self

    def transform(self, X):
        X_copy = X.copy()
        to_drop = list(set(self.cols_to_remove + self.high_nan_cols))
        X_copy = X_copy.drop(columns=[c for c in to_drop if c in X_copy.columns])
        return X_copy.fillna(-999)

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, group_cols=['card1', 'addr1']):
        self.group_cols = group_cols
        self.group_mapping = {}

    def fit(self, X, y=None):
        for col in self.group_cols:
            if col in X.columns:
                self.group_mapping[col] = X.groupby(col)['TransactionAmt'].agg(['mean', 'std']).to_dict()
        return self

    def transform(self, X):
        X_out = X.copy()
        if 'TransactionDT' in X.columns:
            X_out['dt_hour'] = (X_out['TransactionDT'] // 3600) % 24
            X_out['dt_day'] = (X_out['TransactionDT'] // (3600 * 24)) % 7
        
        for col, stats in self.group_mapping.items():
            X_out[f'{col}_amt_mean'] = X_out[col].map(stats['mean'])
            X_out[f'{col}_amt_std'] = X_out[col].map(stats['std'])
        
        return X_out.fillna(0)

class FraudEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.cat_features = []

    def fit(self, X, y=None):
        self.cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.cat_features:
            self.oe.fit(X[self.cat_features].astype(str))
        return self

    def transform(self, X):
        X_copy = X.copy()
        if self.cat_features:
            X_copy[self.cat_features] = self.oe.transform(X_copy[self.cat_features].astype(str))
        return X_copy

Overwriting my_preprocessing_classes.py
